In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd drive/MyDrive/7880/

Mounted at /content/drive


## Libraries

In [ ]:
import os
import glob
import json
import argparse
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

## Helper Functions

In [5]:
def get_average_class_embedding(class_folder_path):
    """
    Loads all .npy image embeddings (common + counter) for a specific class
    and returns the mean embedding.
    """
    image_files = []
    for root, dirs, files in os.walk(class_folder_path):
        for file in files:
            if file.endswith('.npy'):
                image_files.append(os.path.join(root, file))

    if not image_files:
        return None

    # Load all images
    embeddings = []
    for f in image_files:
        embeddings.append(np.load(f))

    emb_matrix = torch.tensor(np.array(embeddings))
    emb_matrix = emb_matrix / emb_matrix.norm(dim=-1, keepdim=True)

    avg_emb = torch.mean(emb_matrix, dim=0)
    avg_emb = avg_emb / avg_emb.norm(dim=-1, keepdim=True)
    return avg_emb

def load_text_embeddings(model_text_path):
    """
    Loads 1000 ImageNet text embeddings.
    Returns: Tensor [1000, Dim] on correct device.
    """
    files = glob.glob(os.path.join(model_text_path, "*.npy"))
    embeddings = [None] * 1000

    if not files:
        raise FileNotFoundError(f"No npy files found in {model_text_path}")

    print(f"Loading text embeddings...")
    for f in files:
        filename = os.path.basename(f)
        try:
            class_id = int(filename.split('_')[0])
            embeddings[class_id] = np.load(f)
        except Exception:
            continue

    if any(x is None for x in embeddings):
        raise ValueError("Missing some ImageNet text classes (0-999). Check data.")

    emb_tensor = torch.tensor(np.array(embeddings)).float()
    # Normalize
    emb_tensor = emb_tensor / emb_tensor.norm(dim=-1, keepdim=True)
    return emb_tensor

def load_images_from_split(folder_path):
    """
    Loads all .npy files in a specific split folder (recursively).
    Returns: Tensor [N, Dim]
    """
    paths = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.npy'):
                paths.append(os.path.join(root, file))

    if not paths:
        return None

    embs = [np.load(p) for p in paths]
    embs_tensor = torch.tensor(np.array(embs)).float()
    embs_tensor = embs_tensor / embs_tensor.norm(dim=-1, keepdim=True)
    return embs_tensor

def evaluate_split(image_tensor, text_tensor, gt_id, setup, candidate_indices=None):
    """
    Computes accuracy for a batch of images against text embeddings.

    Args:
        image_tensor: [N, D]
        text_tensor: [1000, D]
        gt_id: int (Ground Truth Class ID)
        setup: '1vs1000' or '1vs20'
        candidate_indices: list of ints (required if setup is 1vs20)

    Returns:
        accuracy (float)
    """
    if image_tensor is None:
        return np.nan

    device = image_tensor.device

    # Compute Logits: [N, 1000]
    logits = torch.matmul(image_tensor, text_tensor.t())

    if setup == '1vs20':
        if candidate_indices is None:
            raise ValueError("Candidate indices missing for 1vs20")

        mask = torch.ones_like(logits) * -float('inf')
        indices_tensor = torch.tensor(candidate_indices, device=device)


        mask[:, indices_tensor] = logits[:, indices_tensor]
        predictions = torch.argmax(mask, dim=1)
    else:
        # 1vs1000
        predictions = torch.argmax(logits, dim=1)

    correct = (predictions == gt_id).sum().item()
    total = image_tensor.shape[0]

    return correct / total

def load_images_from_split_fast(folder_path, num_workers=4):
    """
    Multithreaded loader for .npy files to speed up I/O.
    """
    paths = []
    for root, _, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.npy'):
                paths.append(os.path.join(root, file))

    if not paths:
        return None

    def load_npy(p):
        return np.load(p)

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        embs = list(executor.map(load_npy, paths))

    embs_tensor = torch.tensor(np.array(embs)).float()
    embs_tensor = embs_tensor / embs_tensor.norm(dim=-1, keepdim=True)
    return embs_tensor

## Main Loop

In [ ]:
SETUP = '1vs1000'  # Options: '1vs1000' or '1vs20'
EMBEDDINGS_ROOT = 'embeddings'
CONFUSING_CLASSES_PATH = 'confusing_classes.json'
RESULTS_DIR = 'results'
NUM_WORKERS = 4  # For loading .npy files in parallel

models_list = [
    "openai_vit_b32"
]

os.makedirs(RESULTS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

mapping = None
if SETUP == '1vs20':
    if os.path.exists(CONFUSING_CLASSES_PATH):
        with open(CONFUSING_CLASSES_PATH, 'r') as f:
            mapping = {int(k): v for k, v in json.load(f).items()}
    else:
        raise FileNotFoundError(f"Could not find {CONFUSING_CLASSES_PATH}")


print(f"Starting evaluation for {len(models_list)} models in {SETUP} mode...\n")

for model_name in tqdm(models_list, desc="Models"):

    out_file = os.path.join(RESULTS_DIR, f"results_{model_name}_{SETUP}.csv")
    if os.path.exists(out_file):
        print(f"Skipping {model_name}: Results file already exists ({out_file}).")
        continue

    text_path = os.path.join(EMBEDDINGS_ROOT, "text_embeddings", model_name)
    img_root_path = os.path.join(EMBEDDINGS_ROOT, "image_embeddings", model_name)

    if not os.path.exists(text_path) or not os.path.exists(img_root_path):
        print(f"Skipping {model_name}: Embeddings folder not found.")
        continue

    try:
        text_emb = load_text_embeddings(text_path).to(device)
    except Exception as e:
        print(f"Failed to load text embeddings for {model_name}: {e}")
        continue

    results = []
    try:
        class_folders = sorted(os.listdir(img_root_path))
    except FileNotFoundError:
        continue

    for class_dir in tqdm(class_folders, desc=f"Classes ({model_name})", leave=False):
        full_class_path = os.path.join(img_root_path, class_dir)
        if not os.path.isdir(full_class_path):
            continue

        # Parse Ground Truth ID
        try:
            parts = class_dir.split(' ', 1)
            gt_id = int(parts[0])
            class_name = parts[1] if len(parts) > 1 else str(gt_id)
        except ValueError:
            continue

        candidates = mapping.get(gt_id) if SETUP == '1vs20' else None

        # Traverse subfolders to find 'common' and 'counter'
        easy_tensors = []
        hard_tensors = []

        subfolders = os.listdir(full_class_path)

        for sub in subfolders:
            sub_path = os.path.join(full_class_path, sub)
            if not os.path.isdir(sub_path): continue

            if sub.lower().startswith('common'):
                t = load_images_from_split_fast(sub_path, num_workers=NUM_WORKERS)
                if t is not None: easy_tensors.append(t)
            elif sub.lower().startswith('counter'):
                t = load_images_from_split_fast(sub_path, num_workers=NUM_WORKERS)
                if t is not None: hard_tensors.append(t)

        if easy_tensors:
            final_easy = torch.cat(easy_tensors).to(device)
            acc_e = evaluate_split(final_easy, text_emb, gt_id, SETUP, candidates)
        else:
            acc_e = np.nan

        if hard_tensors:
            final_hard = torch.cat(hard_tensors).to(device)
            acc_h = evaluate_split(final_hard, text_emb, gt_id, SETUP, candidates)
        else:
            acc_h = np.nan

        results.append({
            "Class ID": gt_id,
            "Class Name": class_name,
            "Easy Accuracy": acc_e,
            "Hard Accuracy": acc_h
        })

    if results:
        df = pd.DataFrame(results)
        df["Drop"] = df["Easy Accuracy"] - df["Hard Accuracy"]

        # Stats
        avg_easy = df["Easy Accuracy"].mean()
        avg_hard = df["Hard Accuracy"].mean()
        avg_drop = df["Drop"].mean()

        print(f"  -> Easy: {avg_easy:.4f} | Hard: {avg_hard:.4f} | Drop: {avg_drop:.4f}")

        df.to_csv(out_file, index=False)
    else:
        print(f"  -> No results found for {model_name}")